In [ ]:
%pip install psxdata
dbutils.library.restartPython()

In [ ]:
import psxdata
from pyspark.sql import SparkSession

tickers = ["UBL", "MCB", "MEBL", "ENGRO", "LUCK"]

pdf_list = []
for t in tickers:
    df = psxdata.stocks(t, start="2024-01-01", end="2026-08-18")
    df["symbol"] = t
    pdf_list.append(df)

import pandas as pd
raw_pdf = pd.concat(pdf_list)

# Convert to Spark and write as a Delta (Bronze) table
bronze_df = spark.createDataFrame(raw_pdf)
bronze_df.write.format("delta").mode("overwrite").saveAsTable("psx_bronze")

In [ ]:
%sql
SELECT * FROM psx_bronze LIMIT 10

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze = spark.table("psx_bronze")

w = Window.partitionBy("symbol").orderBy("date")

silver = (
    bronze
    .dropDuplicates(["symbol", "date"])
    .withColumn("daily_return_pct", 
        (F.col("close") - F.lag("close").over(w)) / F.lag("close").over(w) * 100)
    .withColumn("ma_20", F.avg("close").over(w.rowsBetween(-19, 0)))
)

silver.write.format("delta").mode("overwrite").saveAsTable("psx_silver")

In [ ]:
gold = (
    spark.table("psx_silver")
    .groupBy("symbol")
    .agg(
        F.max("date").alias("latest_date"),
        F.round(F.last("close"), 2).alias("latest_close"),
        F.round(F.avg("daily_return_pct"), 3).alias("avg_daily_return_pct"),
        F.round(F.stddev("daily_return_pct"), 3).alias("volatility")
    )
)

gold.write.format("delta").mode("overwrite").saveAsTable("psx_gold")
display(gold)